In [ ]:
import bilby
from bilby.core.utils import random
import pprint
import copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import bilby.gw.conversion as conv
from bilby.hyper.likelihood import HyperparameterLikelihood

random.seed(123)

In [ ]:
# TODO: 
    # - choose N events
    # - set up masses and lambdas following MPA1 EOS
    # - choose luminosity distances for events
N = 10
masses = None
lambdas = None
distances = None

results = []
bilby_priors = [] # save the bilby priors used in each run to use for reweighting later on

# TODO: 
# - replace with SLURM array run feature (jobs can go in parallel)
for i in range(N):
    outdir = "outdir"
    label_raw = "bns_eos"
    label = label_raw + str(i)
    bilby.utils.check_directory_exists_and_if_not_mkdir(outdir)

    #everything except masses, lambdas, distance
    ip_template = dict(
        chi_1=0.02,
        chi_2=0.02,
        theta_jn=0.4,
        psi=2.659,
        phase=1.3,
        geocent_time=1126259642.413,
        ra=1.375,
        dec=-1.2108,
    )

    waveform_arguments = dict(
        waveform_approximant="IMRPhenomPv2_NRTidal",
        reference_frequency=50.0, #freq at which spin and orbital angles are defined
        minimum_frequency=10.0, #to match min freq 10 for CE
        fiducial=1
    )

    duration = 32
    sampling_frequency = 2048
    start_time = ip_template["geocent_time"] + 2 - duration


    waveform_generator = bilby.gw.WaveformGenerator(
        duration=duration,
        sampling_frequency=sampling_frequency,
        frequency_domain_source_model=bilby.gw.source.lal_binary_neutron_star_relative_binning,
        parameter_conversion=bilby.gw.conversion.convert_to_lal_binary_neutron_star_parameters,
        waveform_arguments=waveform_arguments,
    )

    #build ip
    ip = ip_template.copy()
    ip["mass_2"], ip["mass_1"] = masses[2*i], masses[2*i+1]
    ip["lambda_2"], ip["lambda_1"] = lambdas[2*i], lambdas[2*i+1]
    ip["luminosity_distance"] = distances[i]

    ifo = bilby.gw.detector.InterferometerList(["CE"])
    for interferometer in ifo:
        interferometer.minimum_frequency = 10

    ifo.set_strain_data_from_power_spectral_densities(sampling_frequency=sampling_frequency, duration=duration, start_time=start_time)
    ifo.inject_signal(parameters=ip, waveform_generator=waveform_generator)

    priors = bilby.gw.prior.BNSPriorDict()

    # note: if lum_dist removed from priors, make sure distance_marginalization = True in RelativeBinningGravitationalWaveTransient likelihood
    for key in [
        "psi",
        "geocent_time",
        "ra",
        "dec",
        "chi_1",
        "chi_2",
        "theta_jn",
        "luminosity_distance",
        "phase",
    ]:
        priors[key] = ip[key]

    del priors["mass_ratio"], priors["lambda_1"], priors["lambda_2"]

    m1, m2 = ip["mass_1"], ip["mass_2"]
    chirp_mass = bilby.gw.conversion.component_masses_to_chirp_mass(m1, m2)

    # priors["chirp_mass"] =  bilby.core.prior.Uniform(0.5, 1.5, name="chirp_mass", unit="$M_{\\odot}$")

    priors["chirp_mass"] = bilby.core.prior.Gaussian(chirp_mass, 0.1, name="chirp_mass", unit="$M_{\\odot}$")

    priors["symmetric_mass_ratio"] = bilby.core.prior.Uniform(0.1, 0.25, name="symmetric_mass_ratio")
    priors["lambda_tilde"] = bilby.core.prior.Uniform(0, 2500, name="lambda_tilde")
    priors["delta_lambda_tilde"] = bilby.core.prior.Uniform(-500, 1000, name="delta_lambda_tilde")

    bilby_priors.append({
        "chirp_mass": priors["chirp_mass"],
        "symmetric_mass_ratio": priors["symmetric_mass_ratio"],
        "lambda_tilde": priors["lambda_tilde"],
        "delta_lambda_tilde": priors["delta_lambda_tilde"],
    })


    priors["lambda_1"] = bilby.core.prior.Constraint(name="lambda_1", minimum=0, maximum=5000)
    priors["lambda_2"] = bilby.core.prior.Constraint(name="lambda_2", minimum=0, maximum=5000)

    fp = ip.copy()

    m1 = fp.pop("mass_1")
    m2 = fp.pop("mass_2")
    l1 = fp.pop("lambda_1")
    l2 = fp.pop("lambda_2")

    fp["chirp_mass"] = (bilby.gw.conversion.component_masses_to_chirp_mass(m1, m2))
    fp["symmetric_mass_ratio"] = (bilby.gw.conversion.component_masses_to_symmetric_mass_ratio(m1, m2))
    fp["lambda_tilde"] = (bilby.gw.conversion.lambda_1_lambda_2_to_lambda_tilde(l1, l2, m1, m2))
    fp["delta_lambda_tilde"] = (bilby.gw.conversion.lambda_1_lambda_2_to_delta_lambda_tilde(l1, l2, m1, m2))


    likelihood = bilby.gw.likelihood.RelativeBinningGravitationalWaveTransient(
        interferometers=ifo,
        waveform_generator=waveform_generator,
        priors=priors,
        fiducial_parameters=fp,
    )

    result = bilby.run_sampler(
        likelihood=likelihood,
        priors=priors,
        sampler="nestle",
        npoints=100,
        injection_parameters=ip,
        outdir=outdir,
        label=label,
        conversion_function=bilby.gw.conversion.generate_all_bns_parameters,
        result_class=bilby.gw.result.CBCResult,
    )

    print("finished merger", i)
    results.append(result)

In [ ]:
# hyper-parameter inference

# define hyper-prior -> use narrow gaussian around EOS predicted values of lambda_1, lambda_2
# hyper-params are param1, param2, param3, log10_pressure1_cgs, log10_pressure2_cgs
def hyper_prior(dataset, param1, param2, param3, log10_pressure1_cgs, log10_pressure2_cgs):
    chirp_mass = dataset["chirp_mass"]
    eta = dataset["symmetric_mass_ratio"]
    lambda_tilde = dataset["lambda_tilde"]
    delta_lambda_tilde = dataset["delta_lambda_tilde"]

    mass_1, mass_2 = conv.chirp_mass_and_mass_ratio_to_component_masses(
        chirp_mass, conv.symmetric_mass_ratio_to_mass_ratio(eta)
    )

    lambda_1_pred, lambda_2_pred, eos_check = conv.polytrope_or_causal_params_to_lambda_1_lambda_2(
        param1, log10_pressure1_cgs, param2, log10_pressure2_cgs, param3,
        mass_1, mass_2, causal=0
    )

    lambda_tilde_pred = conv.lambda_1_lambda_2_to_lambda_tilde(lambda_1_pred, lambda_2_pred, mass_1, mass_2)
    delta_lambda_tilde_pred = conv.lambda_1_lambda_2_to_delta_lambda_tilde(lambda_1_pred, lambda_2_pred, mass_1, mass_2)

    # TODO: tune this
    kernel_sigma = 1.0  
    prob = (
        np.exp(-0.5 * ((lambda_tilde - lambda_tilde_pred) / kernel_sigma) ** 2)
        * np.exp(-0.5 * ((delta_lambda_tilde - delta_lambda_tilde_pred) / kernel_sigma) ** 2)
    )

    return np.where(eos_check, prob, 0.0)


samples = [result.posterior for result in results]

for i in range(len(samples)):
    sample = samples[i]

    # prior value will be the product of the prior prob. for each of the params used in the hyperprior -> ie. chirp_mass, eta, lambda_tilde, delta_lambda_tilde
    # use bilby_priors (which sav)

    sample["prior"] = 1
    for name in bilby_priors[i]: # name runs over the four params used in the hyperprior
        sample["prior"] *= bilby_priors[name].prob(sample[name])


evidences = [result.log_evidence for result in results]

hp_likelihood = HyperparameterLikelihood(
    posteriors=samples,
    hyper_prior=hyper_prior,
    log_evidences=evidences,
    max_samples=500,
)

# TODO: find motivated priors for these PP params
hp_priors = dict(
    param1=None,
    param2=None,
    param3=None,
    log10_pressure1_cgs=None,
    log10_pressure2_cgs=None,
)

result = bilby.run_sampler(
    likelihood=hp_likelihood,
    priors=hp_priors,
    sampler="dynesty",
    nlive=1000,
    use_ratio=False,
    outdir=outdir,
    label="hyper_eos",
    verbose=True,
    clean=True,
)

result.plot_corner()